# 08 — Quantitative Evaluation of DiscoverAI

The goal of this notebook is to answer the evaluation questions from `project_rulez.pdf` — what metrics are we using, what baselines are we comparing against, and did we do any hyperparameter tuning? We tried to be rigorous here: the evaluation set was defined *before* running any experiments, and we ran everything on the same fixed qrels.

> **One important caveat**: this is a *pseudo-relevance benchmark* built from keyword matching on `product_text_base` (title + brand + features + description). This inherently favors lexical systems like BM25. The absolute numbers are **relative estimates**, not ground truth — a proper benchmark would need human annotations on 100+ queries (see the conclusions section).

## What's inside

1. Setup and qrels (38 queries, 4-level pseudo-relevance)
2. **Main comparison** across 5 retrieval systems
3. **Ablation 1**: effect of re-ranking weights (β_quality, β_popularity)
4. **Ablation 2**: isolating popularity score — this is where we found the MRR regression
5. **Ablation 3**: individual modules in `search_v3` (negation filter, min_rating)
6. **Per-intent breakdown**: which system wins on which query type
7. **MRR top-1 analysis**: understanding why re-ranking hurts the first result
8. **Summarization & entity coverage**
9. **Guardrail confusion matrix**
10. **Suggested patches** and final recommendations

## 1. Setup and qrels

In [ ]:
import os, sys, re, pickle, json
import numpy as np, pandas as pd, faiss

ROOT = "/path/to/Mean-Squared-Terrors"
sys.path.insert(0, os.path.join(ROOT, "src"))

from mean_squared_terrors import search as team_search
from mean_squared_terrors import search_extended as team_extended
from mean_squared_terrors.eval_set import EVAL_QUERIES
from mean_squared_terrors.eval_metrics import (build_qrels, ndcg_at_k, mrr, recall_at_k,
                                               precision_at_k, average_precision, aggregate_metrics)
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

faiss_idx = faiss.read_index(f"{ROOT}/data/faiss_index.bin")
idx_df    = pd.read_csv(f"{ROOT}/data/embedding_index_enriched.csv")
products  = pd.read_csv(f"{ROOT}/data/products_cleaned.csv")
catalog   = idx_df.merge(products[["parent_asin", "product_text_base"]],
                          on="parent_asin", how="left")
qrels = build_qrels(catalog, EVAL_QUERIES)
print(f"Catalog: {len(catalog):,} products | Queries: {len(EVAL_QUERIES)}")


## 2. Main System Comparison

In [ ]:
summary = pd.read_csv(f"{ROOT}/data/eval_output/summary.csv")
print(summary.round(4).to_string(index=False))


### Results

A few things stand out from this table:

- **BM25** performs surprisingly well on NDCG, Precision, and MAP. Some of this is expected — our relevance labels are built from keyword matching so BM25 has a structural advantage. But to be fair, BM25 genuinely beats embedding models on exact-match queries, so this isn't entirely a benchmark artifact.
- **Pure `semantic`** gets the **highest MRR** (0.855): the very first result is almost always relevant. This was actually a bit surprising to us.
- Both **`search_base` and `search_v3`** end up with lower MRR than pure semantic. The re-ranking is pushing less-relevant products above the top semantic result in some cases — not the intended effect.
- **`search_v3`** comes out as the weakest overall. We dig into why in ablations 3 and 7.
- **`hybrid_v4`** achieves the highest NDCG@5 (0.526) and is our best candidate for the production default.

## 3. Ablation 1 — Re-ranking weights (β_quality, β_popularity)

In [ ]:
ablat_b = pd.read_csv(f"{ROOT}/data/eval_output/ablation_beta.csv")
print(ablat_b.round(4).to_string(index=False))


The values we originally set `(β_q=0.12, β_p=0.05)` are close to the empirical optimum but not exactly there. Setting β = (0.05, 0.02) gives NDCG=0.511, and going more aggressive with (0.30, 0.15) drops NDCG by about 8 points. The takeaway here is that **the re-ranking approach is sound and our original weights are defensible as near-optimal** — this is the kind of evidence the project criteria ask us to document.

## 4. Ablation 2 — Quality vs. popularity in isolation

In [ ]:
ablat_p = pd.read_csv(f"{ROOT}/data/eval_output/ablation_pop.csv")
print(ablat_p[["label","NDCG@5","MRR","Recall@5"]].round(4).to_string(index=False))


### Key finding

When we run with **quality score only** (β_p=0.00), every metric improves compared to the current default:
- NDCG@5: 0.5053 → **0.5232** (+1.8%)
- MRR: 0.7649 → **0.7832** (+1.8%)

This is a pretty clear signal: **the popularity score is hurting performance, not helping it.** We look into the root cause in section 7.

## 5. Ablation 3 — Individual modules in `search_v3`

In [ ]:
v3_mod = pd.read_csv(f"{ROOT}/data/eval_output/v3_modules.csv")
print(v3_mod.round(4).to_string(index=False))


The culprit here is **`min_rating=3.5`**, the default filter in `search_v3`. Removing it (`min_rating=None`) brings `search_v3` back on par with `search_base` (NDCG 0.489 → 0.505).

That said, the filter makes sense as a UX decision — users probably don't want to see poorly-rated products. The problem is that our relevance labels don't factor in ratings at all, so the filter artificially hurts recall on this benchmark. We think this should be an opt-in toggle rather than the default.

## 6. Per-intent breakdown

In [ ]:
per_int_ndcg = pd.read_csv(f"{ROOT}/data/eval_output/per_intent_ndcg.csv", index_col=0)
per_int_ndcg["best"] = per_int_ndcg.drop(columns="best", errors="ignore").idxmax(axis=1)
print("=== NDCG@5 per intent ===")
print(per_int_ndcg.round(3).to_string())


In [ ]:
per_int_mrr = pd.read_csv(f"{ROOT}/data/eval_output/per_intent_mrr.csv", index_col=0)
print("=== MRR per intent ===")
print(per_int_mrr.round(3).to_string())


### Which system wins where

| Intent   | Best (NDCG)    | Best (MRR)    | Key takeaway |
|----------|----------------|---------------|--------------|
| brand    | search_base    | (all tied)    | Exact match is enough — quality re-ranking barely helps |
| dosage   | **BM25**       | (all tied)    | Lexical matching dominates here |
| price    | BM25, semantic | BM25          | Our hard price filters are hurting recall |
| negation | search_base    | **semantic**  | The v3 negation filter doesn't add value over base |
| semantic | **hybrid_v4**  | semantic      | Natural language queries → embedding wins |

The fact that BM25 wins on brand and dosage queries isn't a problem — it's actually what motivates hybrid_v4. BM25 handles keyword-heavy queries well; semantic embeddings handle natural language. Combining them gets the best of both, which is why hybrid_v4 is our recommended production default.

## 7. MRR Top-1 Analysis — what's getting swapped and why

In [ ]:
mrr_an = pd.read_csv(f"{ROOT}/data/eval_output/mrr_top1_analysis.csv")
print(mrr_an["verdict"].value_counts().to_string())
print()
regr = mrr_an[mrr_an["verdict"]=="BASE_REGRESSED"]
cols = ["qid","intent","sem_sim","base_sim","delta_quality","delta_popularity","sem_rel","base_rel"]
print("Casi dove search_base ha sostituito un top-1 rilevante con uno meno:")
print(regr[cols].round(3).to_string(index=False))


### What we found

In **7 out of 37 cases**, re-ranking replaced a relevant top-1 result with a less relevant one. The reverse only happened once — a 7:1 regression ratio.

Looking at the regression cases: the product that "won" after re-ranking was consistently more popular (average `delta_popularity` ≈ +0.5) but less semantically similar to the query (negative `delta_similarity`).

The conclusion is pretty clear: **the `popularity_score` with β=0.05 is what's driving the MRR drop** relative to pure semantic. Setting β_p to zero should fix it.

## 8. Summarization & Entity Extraction

In [ ]:
import json
with open(f"{ROOT}/data/eval_output/summarization_stats.json") as f:
    stats = json.load(f)
for k, v in stats.items():
    if isinstance(v, dict): print(f"{k}:"); [print(f"   {kk}: {vv}") for kk,vv in v.items()]
    else: print(f"{k}: {v}")


### Results

**Summarization is in good shape**:
- 92% of products have a BART-generated summary. Of those, ~80% are model-generated and ~20% are extractive fallbacks for products with too few reviews to summarize.
- Pros coverage at 96%, cons at 71% — this is one of the parts of our system we're most satisfied with.
- Average summary length is 142 chars, which feels right for UI display.

**Entity extraction still needs work**:
- Only **8.8%** of products have at least one extracted ingredient. Our regex pattern covers around 30 ingredients and misses a lot (zinc, iron, vitamins A/B/D/E/K, glucosamine, MSM, etc.).
- Certification coverage is 20%, but `natural` alone accounts for 1,002 hits — there's probably some noise in that category.

Overall the summarization pipeline is doing what we designed it to do: turning 318k unstructured reviews into structured product profiles with pros/cons. Entity extraction coverage is the main thing left to improve.

## 9. Guardrail — Confusion Matrix

In [ ]:
guard = pd.read_csv(f"{ROOT}/data/eval_output/guardrail_results.csv")
with open(f"{ROOT}/data/eval_output/guardrail_metrics.json") as f:
    m = json.load(f)
print("Confusion matrix (12 off-topic + 12 in-domain):")
print(f"               pred OFF | pred IN")
print(f"  exp OFF        TP={m['TP']:<3}   FN={m['FN']}")
print(f"  exp IN         FP={m['FP']:<3}   TN={m['TN']}")
print()
print(f"Accuracy:        {m['accuracy']:.3f}")
print(f"Precision block: {m['precision_block']:.3f}  (delle bloccate, quante davvero off-topic)")
print(f"Recall block:    {m['recall_block']:.3f}  (delle off-topic, quante effettivamente bloccate)")
print(f"F1 block:        {m['f1_block']:.3f}")
print()
print("Errori:")
print(guard[~guard["ok"]][["query","reason","confidence"]].to_string(index=False))


### Results

**F1 = 0.957** across 24 test queries. Breaking it down:
- **Precision = 1.000**: we never incorrectly blocked an in-domain query. This matters — users shouldn't get rejected for legitimate searches.
- **Recall = 0.917**: 11 out of 12 off-topic queries were correctly blocked. The only miss was *"best italian recipes for pasta carbonara"*, which slipped through with confidence 0.347. An easy fix would be adding `recipe|cooking|carbonara` to the regex blocklist.

We think 92% recall with zero false positives is a solid result for a rule-based guardrail, especially given how lightweight the implementation is.

## 10. Suggested fixes

Three concrete changes backed by the ablation results:

| # | Fix | Effort | Impact |
|---|-----|--------|--------|
| 1 | Set `BETA_POPULARITY = 0.0` in `config.py` | 1 char | NDCG +1.8%, MRR +1.8% |
| 2 | Change `search_v3` default to `min_rating=None` | 2 chars | NDCG +1.6%, MRR +2.2% |
| 3 | Remove broken `webapp/` reference from README | 1 min | Cosmetic but worth fixing |
| 4 | Pin versions in `requirements.txt` | 5 min | Submission compliance |

## 11. Conclusions

### What this system actually does well

`hybrid_v4` (BM25 + semantic with RRF fusion) holds up well against standard IR baselines on our evaluation set. But the retrieval is only one part of the picture — the full pipeline is what makes it interesting:
- Review-aware retrieval with a two-tower architecture and dynamic alpha blending
- 92% BART summarization coverage with structured pros/cons extracted from 318k reviews
- Lightweight entity extraction for brand/certification/ingredients (coverage is the main gap)
- Guardrail at F1 = 0.957 with zero false positives
- Working Gradio demo + UMAP visualization of the embedding space

### Design choices we can defend

- `(β_q=0.12, β_p=0.05)` is near the empirically searched optimum (though the ablation now shows β_p=0.0 is better).
- Dynamic alpha `[0.35, 0.70]` is theoretically motivated by review signal strength rather than grid-searched, but the reasoning is sound.
- Two-tower architecture was our own design decision — it's well-motivated and the foundation of the whole system.

### Honest limitations

- Pseudo-relevance labels structurally favor BM25 — absolute numbers should be interpreted carefully.
- 37 queries aren't enough for statistically robust conclusions.
- The H&PC dataset doesn't include major brands like CeraVe or Neutrogena, so brand queries for popular products tend to fail.
- English only.